In [ ]:
# =============================================================================
# COLAB UTILITY: Upload bank.csv → Sample 1000 rows → Download new CSV
# =============================================================================
# Run this cell FIRST before anything else.
# It uploads your full bank.csv (4521 rows), randomly picks 1000 rows,
# saves them as bank_sampled_1000.csv, and downloads it to your computer.
#
# Then upload bank_sampled_1000.csv when the main notebook asks for bank.csv.
# =============================================================================

import pandas as pd
import numpy as np
from google.colab import files

# -----------------------------------------------------------------------------
# STEP 1: Upload the CSV file
# -----------------------------------------------------------------------------
print("Please upload bank.csv ...")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print(f"\n✅ Uploaded: {filename}")

# -----------------------------------------------------------------------------
# STEP 2: Read the CSV
# -----------------------------------------------------------------------------
# bank.csv uses semicolon (;) as separator — NOT comma
df = pd.read_csv(filename, sep=';')

print(f"\n📄 Original dataset:")
print(f"   Rows:    {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Column names: {list(df.columns)}")
print(f"\nFirst 3 rows:")
print(df.head(3).to_string())

# -----------------------------------------------------------------------------
# STEP 3: Randomly sample 1000 rows
# -----------------------------------------------------------------------------
SAMPLE_SIZE = 1000
np.random.seed(42)   # fixed seed → same 1000 rows every run

if len(df) <= SAMPLE_SIZE:
    print(f"\n⚠ Dataset only has {len(df)} rows — using all rows.")
    df_sampled = df.copy()
else:
    df_sampled = df.sample(n=SAMPLE_SIZE, random_state=42)
    df_sampled = df_sampled.reset_index(drop=True)

print(f"\n✅ Sampled dataset:")
print(f"   Rows:    {len(df_sampled)}")
print(f"   Columns: {len(df_sampled.columns)}")
print(f"\nPreview of sampled data (first 5 rows):")
print(df_sampled.head(5).to_string())

# Confirm key numeric columns are present
for col in ['age', 'balance', 'duration']:
    print(f"  {col}: min={df_sampled[col].min()}, max={df_sampled[col].max()}, mean={df_sampled[col].mean():.1f}")

# -----------------------------------------------------------------------------
# STEP 4: Save as new CSV (same semicolon format as original)
# -----------------------------------------------------------------------------
output_filename = 'bank_sampled_1000.csv'
df_sampled.to_csv(output_filename, sep=';', index=False)
print(f"\n💾 Saved as: {output_filename}")

# -----------------------------------------------------------------------------
# STEP 5: Download the new CSV to your computer
# -----------------------------------------------------------------------------
print(f"\n⬇ Downloading {output_filename} ...")
files.download(output_filename)
print("✅ Done! Now upload bank_sampled_1000.csv when the main notebook asks.")

# Bank Marketing Dataset — Capacitated K-Center Algorithm
### Euclidean Distance in R³ (age, balance, duration)

**Paper Reference:** "We chose numeric attributes such as age, balance, and duration to represent points in the Euclidean space" (Moro et al., 2014)

**Dataset:** Bank Marketing (4,521 records — 10% sample from 45,211)

**Algorithm:** Same Euclidean-Capacitated-k-Center (Algorithm 1) — only the distance metric changes from Haversine to Euclidean.


In [ ]:
# =============================================================================
# SECTION 1: IMPORTS AND SETUP
# =============================================================================
# This section installs and imports all required libraries for the algorithm.
#
# Libraries used:
#   - numpy: For efficient numerical computations (vectorized distance calculations)
#   - pandas: For data manipulation and loading CSV files
#   - matplotlib/seaborn: For visualization of results
#   - networkx: For constructing and solving the max-flow network (Line 15)
#   - typing: For type hints to improve code readability
# =============================================================================

# Install required packages quietly (-q flag suppresses output)
!pip install networkx matplotlib seaborn pandas numpy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from typing import List, Dict, Tuple, Optional, Set
import warnings
import time

warnings.filterwarnings('ignore')
np.random.seed(42)


In [ ]:
# =============================================================================
# SECTION 2: EUCLIDEAN DISTANCE CALCULATION (VECTORIZED)
# =============================================================================
# This section implements the distance computation required for:
#   - Line 4: R ← { d(c, f) | c ∈ C, f ∈ F }  (all client-facility distances)
#   - Line 12: d(c, f′_i)  (client to center distance for C′_i filtering)
#   - Line 13: d(f, f′_i)  (facility to center distance for F′_i filtering)
#
# EUCLIDEAN DISTANCE is used because our data contains numeric attributes
# (age, balance, duration) in R^d, not geographic coordinates.
#
# Formula: d(a, b) = sqrt( sum_i (a_i - b_i)^2 )
#
# EFFICIENT COMPUTATION using the identity:
#   ||a - b||^2 = ||a||^2 - 2<a,b> + ||b||^2
# This avoids creating an (n1, n2, d) intermediate array and is
# ~10x more memory-efficient for large datasets.
#
# NOTE: In the EMR dataset version, this function used the HAVERSINE formula
# for geographic (lat/lon) distances. Here we use EUCLIDEAN distance because
# the paper datasets use numeric features in R^d.
# =============================================================================

def compute_distance_matrix(points1: np.ndarray, points2: np.ndarray = None) -> np.ndarray:
    """
    Compute pairwise EUCLIDEAN distances between two sets of d-dimensional points.

    This function is used throughout the algorithm for:
    - Line 4: Building the set R of all possible client-facility distances
    - Lines 12-13: Filtering clients and facilities by distance to centers
    - Solution evaluation: Computing actual service distances

    Parameters
    ----------
    points1 : np.ndarray, shape (n1, d)
        First set of points, where each row is a d-dimensional feature vector.
        Typically represents clients (C) or facilities (F).

    points2 : np.ndarray, shape (n2, d), optional
        Second set of points. If None, computes self-distance matrix.

    Returns
    -------
    distances : np.ndarray, shape (n1, n2)
        Distance matrix where distances[i, j] = Euclidean distance
        from points1[i] to points2[j].

    Mathematical Background
    -----------------------
    Euclidean distance: d(a,b) = sqrt( sum_i (a_i - b_i)^2 )

    Efficient expansion:
        ||a - b||^2 = ||a||^2 - 2*a·b + ||b||^2

    This computes ALL n1×n2 distances using matrix operations,
    which is much faster than nested Python loops.

    Example
    -------
    >>> points = np.array([[1, 2, 3], [4, 5, 6]])  # 2 points in 3D
    >>> dist_matrix = compute_distance_matrix(points)
    >>> print(dist_matrix.shape)  # (2, 2)
    """

    if points2 is None:
        points2 = points1

    # ||a||^2 for each point in set 1: shape (n1, 1)
    sq1 = np.sum(points1**2, axis=1)[:, np.newaxis]

    # ||b||^2 for each point in set 2: shape (1, n2)
    sq2 = np.sum(points2**2, axis=1)[np.newaxis, :]

    # a·b dot products: shape (n1, n2)
    dot = points1 @ points2.T

    # ||a - b||^2 = ||a||^2 - 2*a·b + ||b||^2
    dist_sq = sq1 - 2 * dot + sq2

    # Clip to avoid negative values from floating-point errors
    dist_sq = np.clip(dist_sq, 0, None)

    return np.sqrt(dist_sq)


In [ ]:
# =============================================================================
# SECTION 3: PARTITION GENERATION
# =============================================================================
# This section implements Line 5 of the algorithm:
#
#   Line 5: "for each (k₁, k₂, ..., k_τ) such that ∑_{i=1}^{τ} k_i = k"
#
# EXPLANATION:
# After the initial clustering (Lines 1-3), we have τ centers from S′.
# We need to decide how to distribute k facility "slots" among these τ centers.
#
# Each partition (k₁, k₂, ..., k_τ) represents one possible distribution:
#   - k_i = number of facilities to open near the i-th center
#   - The sum of all k_i must equal k (total facilities to open)
#
# EXAMPLE:
#   If k=4 facilities and τ=2 centers, the possible partitions are:
#   [(0,4), (1,3), (2,2), (3,1), (4,0)]
#   - (0,4): 0 facilities at center 1, 4 at center 2
#   - (2,2): 2 facilities at each center
#   etc.
#
# MATHEMATICAL NOTE:
# This is the "stars and bars" combinatorics problem.
# The number of ways to partition k into τ non-negative integers is C(k+τ-1, τ-1).
# For k=5, τ=5: C(9,4) = 126 partitions.
# =============================================================================

def generate_partitions(k: int, tau: int) -> List[List[int]]:
    """
    Generate all ways to partition k facilities among τ (tau) centers.

    This implements Line 5 of the algorithm:
        "for each (k₁, k₂, ..., k_τ) such that ∑_{i=1}^{τ} k_i = k"

    The function uses RECURSIVE enumeration to generate all valid partitions.
    Each partition represents a different strategy for distributing facilities
    among the cluster regions defined by the initial clustering.

    Parameters
    ----------
    k : int
        Total number of facilities to distribute (same k as in the algorithm).
        This is the total capacity we need to allocate across all centers.

    tau : int
        Number of centers from the initial clustering (τ = |S'| from Line 3).
        This is the number of "buckets" to distribute facilities into.

    Returns
    -------
    partitions : List[List[int]]
        List of all valid partitions. Each partition is a list of length tau,
        where partition[i] = k_i = number of facilities assigned to center i.

        Properties of each partition:
        - len(partition) == tau
        - sum(partition) == k
        - all(ki >= 0 for ki in partition)

    Algorithm (Recursive)
    ---------------------
    Base case: If tau == 1, only one partition exists: [k]

    Recursive case: For each possible value i ∈ {0, 1, ..., k}:
        - Assign i facilities to the first center
        - Recursively partition remaining (k - i) facilities among (tau - 1) centers
        - Prepend i to each recursive result

    Time Complexity: O(C(k+tau-1, tau-1)) where C is binomial coefficient

    Examples
    --------
    >>> generate_partitions(4, 2)
    [[0, 4], [1, 3], [2, 2], [3, 1], [4, 0]]

    >>> generate_partitions(3, 3)
    [[0, 0, 3], [0, 1, 2], [0, 2, 1], [0, 3, 0],
     [1, 0, 2], [1, 1, 1], [1, 2, 0],
     [2, 0, 1], [2, 1, 0],
     [3, 0, 0]]
    """

    # -------------------------------------------------------------------------
    # BASE CASE: Only one center remaining
    # -------------------------------------------------------------------------
    # If tau == 1, there's only one way to distribute k facilities:
    # give all k to this single center.
    if tau == 1:
        return [[k]]

    # -------------------------------------------------------------------------
    # RECURSIVE CASE: Distribute facilities across multiple centers
    # -------------------------------------------------------------------------
    # For each possible number of facilities (0 to k) at the first center,
    # recursively find all ways to distribute the remaining facilities
    # among the remaining centers.

    partitions = []  # Will store all valid partitions

    # Try assigning i facilities to the first center, for i = 0, 1, ..., k
    for i in range(k + 1):
        # Recursively get all ways to distribute (k - i) among (tau - 1) centers
        # This is the "divide and conquer" step
        for sub_partition in generate_partitions(k - i, tau - 1):
            # Prepend i to each sub-partition to form the complete partition
            # [i] + [k2, k3, ..., k_tau] = [k1=i, k2, k3, ..., k_tau]
            partitions.append([i] + sub_partition)

    return partitions


# =============================================================================
# DEMONSTRATION: Show how partitions work
# =============================================================================
print(" Partition generation function defined")
print(f"\n   Example: partition(4, 2) = {generate_partitions(4, 2)}")
print(f"\n   Interpretation for k=4, τ=2:")
print(f"   [0, 4] → 0 facilities at center 1, 4 at center 2")
print(f"   [1, 3] → 1 facility at center 1, 3 at center 2")
print(f"   [2, 2] → 2 facilities at each center (balanced)")
print(f"   [3, 1] → 3 facilities at center 1, 1 at center 2")
print(f"   [4, 0] → 4 facilities at center 1, 0 at center 2")
print(f"\n   Number of partitions for k=5, τ=5: {len(generate_partitions(5, 5))}")


In [ ]:
# =============================================================================
# SECTION 4: VANILLA K-CENTER (Initial Clustering)
# =============================================================================
# This section implements Lines 1-3 of the algorithm:
#
#   Line 1: S′, σ′ ← a (1 + ε)-approximate clustering on C, F, k
#   Line 2: r′ ← radius induced by S′, σ′
#   Line 3: τ ← |S′|
#
# ALGORITHM: Farthest-First Traversal (Gonzalez, 1985)
# This is a classic 2-approximation algorithm for k-center:
#   1. Pick an initial center (the one minimizing maximum distance)
#   2. Iteratively add the facility that is FARTHEST from all existing centers
#   3. Repeat until k centers are selected
#
# WHY THIS IS NEEDED:
# The initial clustering provides a "guide" for the capacitated version:
#   - S′: Initial set of center locations (will define regions)
#   - r′: Initial radius (used in Lines 12-13 for distance filtering)
#   - τ: Number of initial centers (determines number of regions)
#
# APPROXIMATION GUARANTEE:
# The farthest-first traversal gives a 2-approximation to optimal k-center.
# This means r′ ≤ 2 × OPT, where OPT is the optimal k-center radius.
# =============================================================================

def vanilla_k_center(C: np.ndarray, F: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray, float, int]:
    """
    Vanilla (uncapacitated) K-Center using Farthest-First Traversal.

    This implements Lines 1-3 of the algorithm and provides a 2-approximation
    to the uncapacitated k-center problem. The results are used to guide
    the capacitated solution.

    Parameters
    ----------
    C : np.ndarray, shape (n, 2)
        Clients array where each row is [latitude, longitude].
        These are the points that need to be served.

    F : np.ndarray, shape (m, 2)
        Facilities array where each row is [latitude, longitude].
        These are the candidate locations for opening centers.

    k : int
        Number of centers to select.

    Returns
    -------
    S_prime : np.ndarray, shape (k, 2)
        Coordinates of the selected center facilities.
        This is S′ from Line 1.

    S_prime_indices : np.ndarray, shape (k,)
        Indices of selected facilities in the original F array.
        S_prime[i] == F[S_prime_indices[i]]

    r_prime : float
        The induced radius = maximum distance from any client to its
        nearest selected center. This is r′ from Line 2.

    tau : int
        Number of selected centers = len(S_prime) = k.
        This is τ from Line 3.

    Algorithm: Farthest-First Traversal
    -----------------------------------
    1. Select first center: Choose the facility that minimizes the maximum
       distance to any client (a "central" facility).

    2. Update distances: For each client, track distance to NEAREST center.

    3. Select next center: Find the client FARTHEST from any current center,
       then open the facility closest to that client.

    4. Repeat step 2-3 until k centers are selected.
    """

    # Get dimensions
    n_clients = len(C)      # Number of clients |C|
    n_facilities = len(F)   # Number of candidate facilities |F|

    # -------------------------------------------------------------------------
    # PRECOMPUTE: Client-to-Facility distance matrix
    # -------------------------------------------------------------------------
    # dist_CF[i, j] = distance from client i to facility j
    # This is used throughout the algorithm for:
    #   - Finding nearest facilities
    #   - Computing service radii
    dist_CF = compute_distance_matrix(C, F)

    # -------------------------------------------------------------------------
    # TRACKING: Minimum distance from each client to ANY selected center
    # -------------------------------------------------------------------------
    # Initially infinite (no centers selected yet)
    # As centers are added, this array is updated via element-wise minimum
    min_dist_to_center = np.full(n_clients, np.inf)

    # List to store indices of selected facility centers
    selected_indices = []

    # =========================================================================
    # STEP 1: Select FIRST center
    # =========================================================================
    # Goal: Choose the facility that is most "central"
    # Strategy: Select facility f that minimizes max_c d(c, f)
    #   - For each facility, find its maximum distance to any client
    #   - Choose the facility with the smallest such maximum

    # max_dists[j] = max distance from facility j to any client
    max_dists = dist_CF.max(axis=0)  # Maximum along client axis

    # Select facility with minimum "max distance to any client"
    first_center = np.argmin(max_dists)

    # Add to selected centers
    selected_indices.append(first_center)

    # Update minimum distances: now each client is dist_CF[c, first_center] away
    min_dist_to_center = np.minimum(min_dist_to_center, dist_CF[:, first_center])

    # =========================================================================
    # STEP 2-3: Greedily add (k-1) more centers using FARTHEST-FIRST
    # =========================================================================
    for _ in range(k - 1):
        # ---------------------------------------------------------------------
        # Find the FARTHEST client from all current centers
        # ---------------------------------------------------------------------
        # This client is currently most "underserved"
        # argmax returns index of maximum element
        farthest_client = np.argmax(min_dist_to_center)

        # Get distances from this farthest client to all facilities
        client_to_facilities = dist_CF[farthest_client, :]

        # ---------------------------------------------------------------------
        # Select the best facility to serve this farthest client
        # ---------------------------------------------------------------------
        # Best = closest facility that hasn't been selected yet
        best_facility = -1
        best_dist = np.inf

        for f_idx in range(n_facilities):
            # Skip already selected facilities
            if f_idx not in selected_indices:
                # Check if this facility is closer to the farthest client
                if client_to_facilities[f_idx] < best_dist:
                    best_dist = client_to_facilities[f_idx]
                    best_facility = f_idx

        # If no unselected facility found, stop (shouldn't happen normally)
        if best_facility == -1:
            break

        # Add the new center to our selection
        selected_indices.append(best_facility)

        # ---------------------------------------------------------------------
        # Update minimum distances for all clients
        # ---------------------------------------------------------------------
        # Each client's min distance = min(current_min, dist_to_new_center)
        # np.minimum performs element-wise minimum
        min_dist_to_center = np.minimum(min_dist_to_center, dist_CF[:, best_facility])

    # =========================================================================
    # PREPARE OUTPUTS (Lines 1-3 results)
    # =========================================================================

    # Line 1: S′ = selected facility coordinates
    S_prime_indices = np.array(selected_indices)  # Array of facility indices
    S_prime = F[S_prime_indices]                   # Actual coordinates of centers

    # Line 2: r′ = radius induced by this clustering
    # = maximum distance from any client to its nearest center
    r_prime = min_dist_to_center.max()

    # Line 3: τ = |S'| = number of centers
    tau = len(S_prime)

    return S_prime, S_prime_indices, r_prime, tau


# =============================================================================
# DEMONSTRATION
# =============================================================================
print(" Vanilla K-Center (Farthest-First Traversal) defined")
print("\n   This function implements Lines 1-3 of the algorithm:")
print("   Line 1: S′, σ′ ← (1+ε)-approximate clustering")
print("   Line 2: r′ ← induced radius (max service distance)")
print("   Line 3: τ ← |S′| (number of initial centers)")

In [ ]:
# =============================================================================
# SECTION 5: FLOW NETWORK CONSTRUCTION (Lines 7-14)
# =============================================================================
# This section builds the flow network used to solve the assignment problem.
# The algorithm uses MAX-FLOW to determine which clients go to which facilities.
#
# ALGORITHM LINES IMPLEMENTED:
#   Line 7:  V ← {s, t} ∪ C  (vertices = source, sink, and all clients)
#   Line 8:  V ← V ∪ ⋃_{i=1}^{τ} k_i copies of i-th facility of S′ (add slots)
#   Line 9:  E ← edges from s to each client with weight 1
#   Line 10: E ← edges from each facility to t with weight U
#   Line 11-14: For each center, compute C'_i, F'_i and add client→slot edges
#
# SLOT-BASED INTERPRETATION:
# Line 8 says to create "k_i copies" of each center. We interpret this as
# creating k_i "slots" per center - abstract nodes representing facility
# capacity. After max-flow, we map slots to actual facilities in F'_i.


def build_flow_network(
    C: np.ndarray,                    # Clients array
    F: np.ndarray,                    # Facilities array
    S_prime_indices: np.ndarray,      # Indices of initial centers in F
    partition: List[int],             # [k1, k2, ..., k_tau] - slots per center
    capacity: int,                    # U - capacity per facility
    r: float,                         # Current radius being tested (from R)
    r_prime: float,                   # Initial clustering radius
    dist_CF: np.ndarray,              # Client-Facility distance matrix
    dist_FF: np.ndarray               # Facility-Facility distance matrix
) -> Tuple[nx.DiGraph, List[str], List[Set[int]], List[Set[int]], List[List[str]], int]:
    """
    Build the flow network for max-flow computation (Lines 7-14).

    This function constructs a directed graph where:
    - Maximum flow from source to sink = number of clients that can be served
    - If max_flow == |C|, all clients can be served within radius r
    - The flow paths determine client→facility assignments

    Parameters
    ----------
    C : np.ndarray, shape (n, 2)
        Client coordinates (latitude, longitude).

    F : np.ndarray, shape (m, 2)
        Facility coordinates (latitude, longitude).

    S_prime_indices : np.ndarray, shape (tau,)
        Indices of the initial centers in F (from vanilla_k_center).
        S'[i] = F[S_prime_indices[i]]

    partition : List[int]
        How to distribute k slots among tau centers.
        partition[i] = k_i = number of slots for center i.
        sum(partition) should equal k.

    capacity : int
        U - maximum clients each facility can serve.
        This is the edge capacity from slots to sink.

    r : float
        The "candidate radius" being tested.
        Clients can only connect to slots where their center is within (r + r') distance.

    r_prime : float
        The radius from initial clustering.
        Used in Lines 12-13 for computing C'_i and F'_i.

    dist_CF : np.ndarray, shape (n, m)
        Precomputed client-facility distances.
        dist_CF[c, f] = distance from client c to facility f.

    dist_FF : np.ndarray, shape (m, m)
        Precomputed facility-facility distances.
        dist_FF[f1, f2] = distance from facility f1 to facility f2.

    Returns
    -------
    G : nx.DiGraph
        The flow network as a NetworkX directed graph.
        Nodes: 'source', 'sink', 'c_0'...'c_{n-1}', 'slot_i_j' for each slot
        Edges: Have 'capacity' attribute for max-flow computation.

    client_nodes : List[str]
        List of client node names ['c_0', 'c_1', ..., 'c_{n-1}'].

    C_prime_sets : List[Set[int]]
        C'_i for each center: clients within (r + r') of center i.
        C_prime_sets[i] = { c_idx : dist(C[c_idx], S'[i]) ≤ r + r' }

    F_prime_sets : List[Set[int]]
        F'_i for each center: facilities within r' of center i.
        F_prime_sets[i] = { f_idx : dist(F[f_idx], S'[i]) ≤ r' }

    slot_nodes : List[List[str]]
        Slot node names organized by center.
        slot_nodes[i] = ['slot_i_0', 'slot_i_1', ..., 'slot_i_{k_i-1}']

    total_slots : int
        Total number of slots created = sum(partition) = k.
    """

    n_clients = len(C)           # Number of clients
    tau = len(S_prime_indices)   # Number of initial centers

    # =========================================================================
    # CREATE DIRECTED GRAPH
    # =========================================================================
    # NetworkX DiGraph supports max-flow algorithms
    G = nx.DiGraph()

    # =========================================================================
    # LINE 7: V ← {s, t} ∪ C
    # =========================================================================
    # Create node names for all clients
    # Format: 'c_0', 'c_1', ..., 'c_{n-1}'
    client_nodes = [f'c_{i}' for i in range(n_clients)]

    # =========================================================================
    # LINE 9: E ← edges from s to each client with weight 1
    # =========================================================================
    # Each client must be assigned to exactly ONE facility
    # Capacity = 1 ensures each client sends at most 1 unit of flow
    #
    # Edge: source ──[capacity=1]──▶ c_i
    #
    for i, cn in enumerate(client_nodes):
        G.add_edge('source', cn, capacity=1)

    # =========================================================================
    # LINES 11-13: Compute C'_i and F'_i for each center
    # =========================================================================
    # For each center f'_i ∈ S':
    #   Line 12: C'_i ← { c ∈ C | d(c, f'_i) ≤ r + r' }
    #            Clients "close enough" to center i (can be served if r is optimal)
    #
    #   Line 13: F'_i ← { f ∈ F | d(f, f'_i) ≤ r' }
    #            Facilities "near" center i (candidates to open in region i)
    #
    # INTUITION:
    # - C'_i: Clients that COULD be served from region i if we achieve radius r
    # - F'_i: Facilities in region i that could serve those clients

    C_prime_sets = []  # Will hold C'_i for each center
    F_prime_sets = []  # Will hold F'_i for each center

    for s_idx in range(tau):
        # Get the index of this center in F
        center_idx = S_prime_indices[s_idx]

        # -----------------------------------------------------------------
        # LINE 12: C'_i ← { c ∈ C | d(c, f'_i) ≤ r + r' }
        # -----------------------------------------------------------------
        # Find all clients within distance (r + r') of this center
        # dist_CF[:, center_idx] gives distances from all clients to this center
        # np.where returns indices where condition is True
        C_prime_i = set(np.where(dist_CF[:, center_idx] <= r + r_prime)[0])

        # -----------------------------------------------------------------
        # LINE 13: F'_i ← { f ∈ F | d(f, f'_i) ≤ r' }
        # -----------------------------------------------------------------
        # Find all facilities within distance r' of this center
        # These are the candidate facilities for this region
        F_prime_i = set(np.where(dist_FF[:, center_idx] <= r_prime)[0])

        C_prime_sets.append(C_prime_i)
        F_prime_sets.append(F_prime_i)

    # =========================================================================
    # LINE 8: V ← V ∪ ⋃_{i=1}^{τ} k_i copies of i-th facility of S′
    # =========================================================================
    # Create k_i "slots" for each center i
    # A slot represents the capacity to serve up to U clients
    #
    # We use slots instead of actual facilities because:
    # 1. The algorithm needs to "try" different facility combinations
    # 2. Slots abstract away which specific facility is used
    # 3. After max-flow, we map each used slot to a facility in F'_i
    #
    # LINE 10: E ← edges from each facility to t with weight U
    # Each slot can serve at most U clients, so slot→sink capacity = U

    slot_nodes = []    # slot_nodes[i] = list of slots for center i
    total_slots = 0    # Counter for total slots (should equal k)

    for s_idx in range(tau):
        k_i = partition[s_idx]  # Number of slots for this center
        center_slots = []       # Slots for this specific center

        for slot_j in range(k_i):
            # Create unique slot name: 'slot_{center_index}_{slot_number}'
            slot_name = f'slot_{s_idx}_{slot_j}'

            # -----------------------------------------------------------------
            # LINE 10: E ← edges from each facility to t with weight U
            # -----------------------------------------------------------------
            # Connect slot to sink with capacity U
            # Edge: slot_i_j ──[capacity=U]──▶ sink
            G.add_edge(slot_name, 'sink', capacity=capacity)

            center_slots.append(slot_name)
            total_slots += 1

        slot_nodes.append(center_slots)

    # =========================================================================
    # LINE 14: E ← edges for each pair (c ∈ C'_i, f ∈ F'_i) with weight 1
    # =========================================================================
    # For each center i, connect clients in C'_i to slots of center i
    # This ensures clients can only be assigned to nearby regions
    #
    # Note: Original algorithm says (c, f) pairs, but with slots we do (c, slot)
    # The slot→facility mapping happens AFTER max-flow in assign_from_flow()
    #
    # Edge: c_j ──[capacity=1]──▶ slot_i_k (if client j ∈ C'_i)

    for s_idx in range(tau):
        # Skip centers with no slots (k_i = 0)
        if not slot_nodes[s_idx]:
            continue

        # For each client in C'_i (clients near this center)
        for c_idx in C_prime_sets[s_idx]:
            cn = f'c_{c_idx}'  # Client node name

            # Connect client to ALL slots of this center
            # (max-flow will choose which slot to use)
            for slot_name in slot_nodes[s_idx]:
                G.add_edge(cn, slot_name, capacity=1)

    return G, client_nodes, C_prime_sets, F_prime_sets, slot_nodes, total_slots


In [ ]:
# =============================================================================
# SECTION 6: ASSIGNMENT FUNCTION (Line 16)
# =============================================================================
# This section implements Line 16 of the algorithm:
#
#   Line 16: (S, σ) ← assign clients and open facilities according to max-flow result
#
# After running max-flow (Line 15), we have flow values indicating which
# client→slot edges are used. This function:
#   1. Extracts client→slot assignments from the flow
#   2. Maps each used slot to an actual facility in F'_i
#   3. Creates the final client→facility assignments (σ)
#   4. Determines which facilities to open (S)
#
# THREE-PHASE ASSIGNMENT:
# Phase 1: Flow → client_to_slot mapping
# Phase 2: slot → actual facility mapping (using F'_i)
# Phase 3: client → facility final assignment
# =============================================================================

def assign_from_flow(
    flow_dict: Dict,              # Flow solution from nx.maximum_flow
    client_nodes: List[str],      # Client node names ['c_0', ..., 'c_n']
    n_clients: int,               # Total number of clients
    capacity: int,                # U - facility capacity
    partition: List[int],         # [k1, k2, ..., k_tau]
    slot_nodes: List[List[str]],  # Slot names per center
    F_prime_sets: List[Set[int]], # F'_i for each center
    S_prime_indices: np.ndarray,  # Indices of initial centers
    dist_FF: np.ndarray           # Facility-Facility distances
) -> Tuple[Optional[Dict[int, int]], Optional[Dict[int, int]], Optional[Set[int]], Optional[Dict[str, int]], bool]:
    """
    Extract client-facility assignments from max-flow result (Line 16).

    This function interprets the max-flow solution to determine:
    - Which facility each client should go to (σ)
    - Which facilities should be opened (S)

    Parameters
    ----------
    flow_dict : Dict
        Flow dictionary from nx.maximum_flow.
        flow_dict[u][v] = flow on edge u→v.
        We look for flow on client→slot edges to determine assignments.

    client_nodes : List[str]
        List of client node names: ['c_0', 'c_1', ..., 'c_{n-1}'].

    n_clients : int
        Total number of clients that need to be assigned.
        We verify that all clients get an assignment.

    capacity : int
        U - maximum capacity per facility.
        Used to verify capacity constraints.

    partition : List[int]
        The partition [k1, k2, ..., k_tau] used for this flow network.

    slot_nodes : List[List[str]]
        Slot node names organized by center.
        slot_nodes[i] = list of slots for center i.

    F_prime_sets : List[Set[int]]
        F'_i for each center: candidate facilities near each center.
        Used in Phase 2 to map slots to actual facilities.

    S_prime_indices : np.ndarray
        Indices of initial centers in F array.
        Used to sort facilities by distance to their center.

    dist_FF : np.ndarray
        Facility-to-facility distance matrix.
        Used to prefer facilities closer to their center.

    Returns
    -------
    assignments : Dict[int, int] or None
        Final client→facility mapping.
        assignments[c_idx] = f_idx means client c_idx is assigned to facility f_idx.
        None if assignment failed.

    facility_loads : Dict[int, int] or None
        Load (number of clients) at each opened facility.
        facility_loads[f_idx] = number of clients assigned to f_idx.

    opened_facilities : Set[int] or None
        Set of facility indices that are opened.
        |opened_facilities| ≤ k if successful.

    slot_to_facility : Dict[str, int] or None
        Mapping from slot names to facility indices.
        Shows which facility each slot was mapped to.

    success : bool
        True if a valid assignment was found, False otherwise.
        Invalid if: not all clients assigned, capacity exceeded, etc.
    """

    tau = len(partition)  # Number of centers

    # =========================================================================
    # PHASE 1: Extract client → slot assignments from flow
    # =========================================================================
    # The max-flow solution tells us which client→slot edges carry flow.
    # If flow_dict[client_node][slot_node] > 0, client is assigned to that slot.
    #
    # Note: Each client should have exactly one outgoing edge with flow > 0

    client_to_slot = {}  # client_idx → slot_name
    slot_loads = {}      # slot_name → number of clients

    for c_idx, client_node in enumerate(client_nodes):
        # Check if this client node has any outgoing flow
        if client_node in flow_dict:
            # Look through all targets of this client
            for target, flow in flow_dict[client_node].items():
                # We only care about slot targets (not source or other nodes)
                if target.startswith('slot_') and flow > 0:
                    # This client is assigned to this slot
                    client_to_slot[c_idx] = target
                    # Track how many clients each slot has
                    slot_loads[target] = slot_loads.get(target, 0) + 1
                    break  # Each client should only go to one slot

    # -----------------------------------------------------------------
    # VALIDATION: All clients must be assigned
    # -----------------------------------------------------------------
    # If not all clients got a slot, the flow was insufficient
    if len(client_to_slot) != n_clients:
        return None, None, None, None, False

    # =========================================================================
    # PHASE 2: Map slots → actual facilities from F'_i
    # =========================================================================
    # Now we know which slots are used (from slot_loads).
    # We need to decide which actual facility from F'_i each slot represents.
    #
    # Strategy: For each center, assign its used slots to distinct facilities
    #           from F'_i, preferring facilities closer to the center.

    slot_to_facility = {}  # slot_name → facility_index
    used_facilities = set()  # Track which facilities are already used

    for s_idx in range(tau):
        center_idx = S_prime_indices[s_idx]  # Index of center in F
        F_prime_i = F_prime_sets[s_idx]      # Candidate facilities for this center
        center_slots = slot_nodes[s_idx]     # All slots for this center

        # Find which of this center's slots are actually used (have clients)
        used_center_slots = [s for s in center_slots if s in slot_loads]

        if not used_center_slots:
            continue  # No clients assigned to this center's slots

        # -----------------------------------------------------------------
        # Get available facilities (preferably not used by other centers)
        # -----------------------------------------------------------------
        # Try to use facilities not already assigned to other slots
        available = list(F_prime_i - used_facilities)

        # If not enough unique facilities, allow reuse within F'_i
        if len(available) < len(used_center_slots):
            available = list(F_prime_i)

        # If still no facilities available, this configuration fails
        if not available:
            return None, None, None, None, False

        # -----------------------------------------------------------------
        # Sort facilities by distance to center (prefer closer ones)
        # -----------------------------------------------------------------
        # Facilities closer to the center tend to give better overall radius
        available.sort(key=lambda f: dist_FF[f, center_idx])

        # -----------------------------------------------------------------
        # Assign each used slot to a facility
        # -----------------------------------------------------------------
        for slot_idx, slot_name in enumerate(used_center_slots):
            # Use modulo to cycle if we run out of unique facilities
            facility = available[slot_idx % len(available)]
            slot_to_facility[slot_name] = facility
            used_facilities.add(facility)

    # =========================================================================
    # PHASE 3: Create final client → facility assignments
    # =========================================================================
    # Now we can map: client → slot → facility
    #
    # assignments[c_idx] = f_idx means σ(c) = f

    assignments = {}    # Final client→facility mapping
    facility_loads = {} # How many clients each facility serves

    for c_idx, slot_name in client_to_slot.items():
        # Check if the slot was mapped to a facility
        if slot_name not in slot_to_facility:
            return None, None, None, None, False

        # Get the facility for this slot
        f_idx = slot_to_facility[slot_name]

        # Record the assignment
        assignments[c_idx] = f_idx

        # Update facility load count
        facility_loads[f_idx] = facility_loads.get(f_idx, 0) + 1

    # -----------------------------------------------------------------
    # VALIDATION: All clients must have assignments
    # -----------------------------------------------------------------
    if len(assignments) != n_clients:
        return None, None, None, None, False

    # -----------------------------------------------------------------
    # VALIDATION: Capacity constraints must be satisfied
    # -----------------------------------------------------------------
    # Each facility can serve at most U clients
    for f_idx, load in facility_loads.items():
        if load > capacity:
            return None, None, None, None, False

    # =========================================================================
    # SUCCESS: Return the solution
    # =========================================================================
    # S = set of opened facilities (keys of facility_loads)
    # σ = assignments dictionary

    return assignments, facility_loads, set(facility_loads.keys()), slot_to_facility, True


# =============================================================================
# DEMONSTRATION
# =============================================================================
print("\n   This function implements Line 16:")
print("   (S, σ) ← assign clients and open facilities according to max-flow result")
print("\n   Three-phase assignment:")
print("   Phase 1: Flow → client→slot mapping")
print("   Phase 2: slot → actual facility (from F'_i)")
print("   Phase 3: Final client → facility assignments")


In [ ]:
# =============================================================================
# SECTION 7: MAIN ALGORITHM - Euclidean-Capacitated-k-Center
# =============================================================================
# This is the main function that implements the complete Algorithm 1:
#
#   Algorithm 1: Euclidean-Capacitated-k-Center(C, F, U)
#
# The algorithm combines all previous functions in a nested search:
#   - Outer loop (Line 5): Try each partition of k among τ centers
#   - Inner loop (Line 6): Try each candidate radius r from R
#   - For each (partition, r): Build flow network, run max-flow, extract assignment
#   - Return the best solution (smallest maximum distance)
#


def capacitated_k_center(
    C: np.ndarray,                 # Client coordinates
    F: np.ndarray,                 # Facility coordinates
    k: int,                        # Number of facilities to open
    capacity: int,                 # U - capacity per facility
    max_partitions: int = 20,      # Limit partitions to explore (speed optimization)
    max_r_samples: int = 100,      # Sample r values instead of all (speed optimization)
    verbose: bool = True           # Print progress information
) -> Optional[Dict]:
    """
    Main algorithm: Euclidean-Capacitated-k-Center.

    This function finds a set of at most k facilities to open, and assigns
    each client to an open facility, such that:
    - No facility serves more than U (capacity) clients
    - The maximum distance any client travels is minimized

    Parameters
    ----------
    C : np.ndarray, shape (n, 2)
        Client coordinates as [latitude, longitude].
        These are the points that need to be served.

    F : np.ndarray, shape (m, 2)
        Candidate facility coordinates as [latitude, longitude].
        We select at most k of these to open.

    k : int
        Maximum number of facilities to open.
        Constraint: |S| ≤ k where S is the set of opened facilities.

    capacity : int
        U - maximum clients each facility can serve.
        Constraint: ∀f ∈ S, |{c : σ(c) = f}| ≤ U

    max_partitions : int, default=20
        Maximum number of partitions to explore in Line 5 loop.
        Set lower for faster execution, higher for better solutions.

    max_r_samples : int, default=100
        Number of r values to sample from R.
        Using percentiles instead of all unique distances.

    verbose : bool, default=True
        If True, print progress information during execution.

    Returns
    -------
    solution : Dict or None
        If successful, a dictionary containing:
        - 'facilities': Coordinates of opened facilities
        - 'facility_indices': Indices in F of opened facilities
        - 'assignments': Dict mapping client_idx → facility_idx
        - 'actual_max_distance': The achieved maximum distance (radius)
        - 'r_tested': The r value that achieved this solution
        - 'r_prime': Initial clustering radius
        - 'flow_value': Max-flow value (should equal n)
        - 'capacity': The capacity U used
        - 'partition': The partition used
        - 'num_facilities': Number of facilities opened
        - 'total_slots': Total slots (= k)
        - 'facility_loads': Dict mapping facility_idx → number of clients
        - 'all_distances': List of all client-to-facility distances
        - 'slot_to_facility': Mapping from slot names to facility indices

        Returns None if no feasible solution found.

    Algorithm Flow
    --------------
    Lines 1-3: Initial clustering via vanilla_k_center()
    Line 4: Compute distance matrix and sample R values
    Line 5: For each partition (k1, ..., k_tau) with sum = k
    Line 6:   For each r in R (via binary search)
    Lines 7-14:   Build flow network
    Line 15:      Run max-flow
    Line 16:      Extract assignments
    Line 17: Return best solution
    """

    n = len(C)  # Number of clients
    m = len(F)  # Number of candidate facilities
    start_time = time.time()  # Track execution time

    # =========================================================================
    # PRINT CONFIGURATION
    # =========================================================================
    if verbose:
        print(f"\n{'='*70}")
        print(f"{'CAPACITATED K-CENTER (SLOT-BASED)':^70}")
        print(f"{'='*70}")
        print(f"  Clients: {n} | Facilities: {m} | k: {k} | Capacity U: {capacity}")
        print(f"{'='*70}\n")

    # =========================================================================
    # LINES 1-3: Initial Clustering via Vanilla K-Center
    # =========================================================================
    # Line 1: S′, σ′ ← (1+ε)-approximate clustering on C, F, k
    # Line 2: r′ ← radius induced by S′, σ′
    # Line 3: τ ← |S′|
    #
    # This gives us:
    #   - S_prime: Initial center coordinates
    #   - S_prime_indices: Indices of initial centers in F
    #   - r_prime: Initial clustering radius
    #   - tau: Number of initial centers

    if verbose:
        print("[Step 1/4] Computing initial clustering (Lines 1-3)...")

    t0 = time.time()
    S_prime, S_prime_indices, r_prime, tau = vanilla_k_center(C, F, k)

    if verbose:
        print(f"  ✓ r' = {r_prime:.4f} miles, tau = {tau} centers ({time.time()-t0:.2f}s)")

    # =========================================================================
    # LINE 4: R ← { d(c, f) | c ∈ C, f ∈ F }
    # =========================================================================
    # Compute ALL client-facility distances.
    # R is the set of candidate radii to try.
    #
    # OPTIMIZATION: Instead of trying all unique distances (could be n×m),
    # we sample using percentiles. This is much faster and typically
    # captures the important radius values.

    if verbose:
        print(f"\n[Step 2/4] Computing distances (Line 4)...")

    t0 = time.time()

    # dist_CF[i,j] = distance from client i to facility j
    dist_CF = compute_distance_matrix(C, F)

    # dist_FF[i,j] = distance from facility i to facility j
    # Used in Lines 12-13 for computing F'_i
    dist_FF = compute_distance_matrix(F, F)

    # Sample r values using percentiles instead of all unique distances
    # This is much faster while still covering the important range
    all_dists = dist_CF.flatten()       # All n×m distances
    all_dists = all_dists[all_dists > 0]  # Remove zeros

    # Get evenly-spaced percentiles from 0% to 100%
    percentiles = np.percentile(all_dists, np.linspace(0, 100, max_r_samples))
    R = np.unique(percentiles)  # Remove duplicates
    R = np.sort(R)              # Sort ascending

    if verbose:
        print(f"  ✓ Distance matrices computed ({time.time()-t0:.2f}s)")
        print(f"  ✓ Testing {len(R)} r values (sampled from {len(np.unique(all_dists))} unique)")

    # =========================================================================
    # LINE 5: Generate Partitions
    # =========================================================================
    # "for each (k₁, k₂, …, k_τ) such that ∑_{i=1}^{τ} k_i = k"
    #
    # Generate all ways to distribute k facility slots among tau centers.
    # Then prioritize and limit the partitions to test (optimization).

    if verbose:
        print(f"\n[Step 3/4] Generating partitions (Line 5)...")

    # Generate all valid partitions
    all_partitions = generate_partitions(k, tau)

    # OPTIMIZATION: Prioritize balanced partitions
    # Sort by: (number of zeros, max - min) to try balanced partitions first
    # Balanced partitions often give better solutions
    all_partitions.sort(key=lambda p: (sum(1 for x in p if x == 0), max(p) - min(p)))

    # Limit to max_partitions for speed
    partitions_to_test = all_partitions[:max_partitions]

    if verbose:
        print(f"  ✓ Testing {len(partitions_to_test)} of {len(all_partitions)} partitions")

    # =========================================================================
    # LINES 5-17: Main Search Loop
    # =========================================================================
    # Outer loop (Line 5): for each partition
    # Inner loop (Line 6): for each r ∈ R
    #   Lines 7-14: Build flow network
    #   Line 15: Run max-flow
    #   Line 16: Assign clients and facilities
    #   Line 17: Track best solution

    if verbose:
        print(f"\n[Step 4/4] Searching for optimal solution (Lines 5-17)...")

    best_solution = None       # Best solution found so far
    best_max_dist = float('inf')  # Best (smallest) maximum distance
    solutions_found = 0        # Counter for valid solutions

    # -------------------------------------------------------------------------
    # LINE 5: for each partition
    # -------------------------------------------------------------------------
    for p_idx, partition in enumerate(partitions_to_test):
        # Progress indicator
        if verbose:
            pct = (p_idx + 1) / len(partitions_to_test) * 100
            print(f"\r  Progress: {pct:5.1f}% | Partition {p_idx+1}/{len(partitions_to_test)}: {partition}", end="")

        # Validation: partition must sum to k
        if sum(partition) != k:
            continue

        # ---------------------------------------------------------------------
        # LINE 6: for each r ∈ R (via BINARY SEARCH)
        # ---------------------------------------------------------------------
        # OPTIMIZATION: Instead of linear scan, use binary search
        # Goal: Find the smallest r that allows a valid assignment

        left, right = 0, len(R) - 1
        partition_best = None
        partition_best_dist = float('inf')

        while left <= right:
            mid = (left + right) // 2
            r = R[mid]

            try:
                # ---------------------------------------------------------
                # LINES 7-14: Build flow network
                # ---------------------------------------------------------
                G, client_nodes, C_prime_sets, F_prime_sets, slot_nodes, total_slots = \
                    build_flow_network(
                        C, F, S_prime_indices, partition,
                        capacity, r, r_prime, dist_CF, dist_FF
                    )

                # ---------------------------------------------------------
                # LINE 15: Run max-flow on (V, E)
                # ---------------------------------------------------------
                # nx.maximum_flow returns (flow_value, flow_dict)
                # flow_value = total flow from source to sink
                # flow_dict = detailed flow on each edge
                flow_value, flow_dict = nx.maximum_flow(G, 'source', 'sink')

                # Check if all clients can be served (flow = n)
                if flow_value >= n:
                    # -------------------------------------------------
                    # LINE 16: Assign clients and open facilities
                    # -------------------------------------------------
                    assignments, facility_loads, opened_facs, slot_to_fac, success = \
                        assign_from_flow(
                            flow_dict, client_nodes, n, capacity,
                            partition, slot_nodes, F_prime_sets,
                            S_prime_indices, dist_FF
                        )

                    if success:
                        # Calculate actual maximum distance achieved
                        # This is the "cost" of the solution
                        actual_max_dist = max(
                            dist_CF[c_idx, f_idx]
                            for c_idx, f_idx in assignments.items()
                        )

                        num_opened = len(opened_facs)

                        # Verify we're not opening more than k facilities
                        if num_opened <= k:
                            solutions_found += 1

                            # Track best solution for this partition
                            if actual_max_dist < partition_best_dist:
                                partition_best_dist = actual_max_dist
                                partition_best = {
                                    'facilities': np.array([F[f] for f in opened_facs]),
                                    'facility_indices': list(opened_facs),
                                    'assignments': dict(assignments),
                                    'actual_max_distance': actual_max_dist,
                                    'r_tested': r,
                                    'r_prime': r_prime,
                                    'flow_value': flow_value,
                                    'capacity': capacity,
                                    'partition': list(partition),
                                    'num_facilities': num_opened,
                                    'total_slots': total_slots,
                                    'facility_loads': dict(facility_loads),
                                    'all_distances': [dist_CF[c, f] for c, f in assignments.items()],
                                    'slot_to_facility': slot_to_fac
                                }

                            # Binary search: try smaller r (go left)
                            right = mid - 1
                        else:
                            # Too many facilities, need larger r
                            left = mid + 1
                    else:
                        # Assignment failed, need larger r
                        left = mid + 1
                else:
                    # Not enough flow, need larger r
                    left = mid + 1

            except Exception:
                # Error in flow computation, try larger r
                left = mid + 1

        # -----------------------------------------------------------------
        # Update global best solution
        # -----------------------------------------------------------------
        if partition_best and partition_best_dist < best_max_dist:
            best_max_dist = partition_best_dist
            best_solution = partition_best

    # =========================================================================
    # LINE 17: Return best solution
    # =========================================================================
    elapsed = time.time() - start_time

    if verbose:
        print(f"\n\n{'='*70}")
        print(f"  SEARCH COMPLETE in {elapsed:.2f} seconds")
        print(f"  Valid solutions found: {solutions_found}")
        print(f"{'='*70}")

        if best_solution:
            print(f"\n  BEST SOLUTION:")
            print(f"    Partition: {best_solution['partition']}")
            print(f"    Facilities opened: {best_solution['num_facilities']} / k={k}")
            print(f"    Max distance: {best_solution['actual_max_distance']:.4f} miles")
            print(f"    Facility indices: {best_solution['facility_indices']}")

            print(f"\n  CONSTRAINTS:")
            print(f"    [✓] At most k={k} facilities: {best_solution['num_facilities']} ≤ {k}")
            print(f"    [✓] Capacity ≤ {capacity}: max={max(best_solution['facility_loads'].values())}")
            print(f"    [✓] All {n} clients assigned")
        else:
            print(f"\n  No feasible solution found.")
            print(f"  Try increasing k or capacity.")

        print(f"\n{'='*70}\n")

    return best_solution


# =============================================================================
# DEMONSTRATION
# =============================================================================
print("Main algorithm defined (SLOT-BASED, OPTIMIZED)")
print("\n   Algorithm 1: Euclidean-Capacitated-k-Center(C, F, U)")
print("\n   This function orchestrates all components:")
print("   - Lines 1-3: vanilla_k_center() for initial clustering")
print("   - Line 4: compute_distance_matrix() for R")
print("   - Line 5: generate_partitions() for partition enumeration")
print("   - Lines 7-14: build_flow_network() for flow construction")
print("   - Line 15: nx.maximum_flow() for solving")
print("   - Line 16: assign_from_flow() for extracting solution")
print("   - Line 17: Return best (S, σ) found")


In [ ]:
# =============================================================================
# SECTION 8: LOAD AND PREPROCESS BANK MARKETING DATASET
# =============================================================================
# This section loads the Bank Marketing dataset and prepares it for the algorithm.
#
# DATA SOURCE:
#   Bank Marketing dataset (Moro et al., 2014)
#   - Portuguese banking institution direct marketing campaigns
#   - bank.csv: 4,521 records (10% sample from 45,211)
#   - 16 input attributes + 1 output attribute
#
# FEATURE SELECTION (as specified by the paper):
#   - age: client age (numeric)
#   - balance: average yearly balance in euros (numeric)
#   - duration: last contact duration in seconds (numeric)
#   These 3 features define points in R^3 (3-dimensional Euclidean space)
#
# In this application:
#   - Clients (C): Bank customers (data points in feature space)
#   - Facilities (F): Candidate cluster centers (same as client locations)
#   - We assume F = C (any data point could be a center)
#
# CRITICAL: Z-SCORE NORMALIZATION
#   Without normalization, balance (range: -3313 to 71188) would completely
#   dominate age (range: 19 to 87) and duration (range: 4 to 3025).
#   Z-score: x_norm = (x - mean) / std → each feature has mean=0, std=1
# =============================================================================

# -----------------------------------------------------------------------------
# Attempt to load dataset from multiple paths
# -----------------------------------------------------------------------------
data_paths_bank = [
    "bank.csv",
    "../bank.csv",
    "/Users/tadala/Downloads/35/bank.csv"
]

df_bank = None
for path in data_paths_bank:
    try:
        df_bank = pd.read_csv(path, sep=';')
        print(f"Loaded data from: {path}")
        break
    except FileNotFoundError:
        continue

if df_bank is None:
    print("Please upload bank.csv...")
    from google.colab import files
    uploaded = files.upload()
    df_bank = pd.read_csv("bank.csv", sep=';')

print(f"\nInitial records: {len(df_bank)}")
print(f"Columns: {list(df_bank.columns)}")

# -----------------------------------------------------------------------------
# Extract numeric features (as specified by the paper)
# -----------------------------------------------------------------------------
# Paper: "We chose numeric attributes such as age, balance, and duration
#          to represent points in the Euclidean space"

numeric_cols_bank = ['age', 'balance', 'duration']
df_bank_numeric = df_bank[numeric_cols_bank].copy()
df_bank_numeric = df_bank_numeric.dropna()

print(f"Records with complete numeric data: {len(df_bank_numeric)}")
print(f"Features used: {numeric_cols_bank}")

# -----------------------------------------------------------------------------
# Feature normalization (Z-score standardization)
# -----------------------------------------------------------------------------
print(f"\nBefore normalization:")
for col in numeric_cols_bank:
    print(f"  {col:>10}: range [{df_bank_numeric[col].min()}, {df_bank_numeric[col].max()}]")

bank_means = df_bank_numeric.mean()
bank_stds = df_bank_numeric.std()
df_bank_normalized = (df_bank_numeric - bank_means) / bank_stds

print(f"\nAfter Z-score normalization:")
print(f"  All features: mean ≈ 0, std ≈ 1")

# -----------------------------------------------------------------------------
# Subsample for computational feasibility
# -----------------------------------------------------------------------------
# With 4521 points, max-flow iterations can be slow.
# We subsample to 1000 points (similar scale to EMR dataset: 873 points).

BANK_SAMPLE_SIZE = 1000
np.random.seed(42)

if len(df_bank_normalized) > BANK_SAMPLE_SIZE:
    sample_indices = np.random.choice(
        len(df_bank_normalized), BANK_SAMPLE_SIZE, replace=False
    )
    X_bank = df_bank_normalized.iloc[sample_indices].values
    print(f"\nSubsampled: {len(df_bank_normalized)} → {BANK_SAMPLE_SIZE} points")
else:
    X_bank = df_bank_normalized.values
    print(f"\nUsing all {len(X_bank)} points (no subsampling needed)")

# C = F (clients and facilities at same locations)
C = X_bank.copy()
F = X_bank.copy()

print(f"\nFinal dataset:")
print(f"  C (clients):    {C.shape[0]} points in {C.shape[1]}D Euclidean space")
print(f"  F (facilities): {F.shape[0]} candidate locations")
print(f"  Distance metric: Euclidean (normalized units)")


In [ ]:
# =============================================================================
# SECTION 9: RUN THE ALGORITHM
# =============================================================================
# Execute the Euclidean-Capacitated-k-Center algorithm on Bank Marketing data.
#
# PARAMETER SELECTION:
#   K = 5: Open 5 facilities (same as EMR experiment for comparison)
#   CAPACITY = 250: Each facility serves up to 250 clients
#     → Total capacity = 5 × 250 = 1250 ≥ 1000 clients ✓
#   MAX_PARTITIONS = 10: Explore top 10 balanced partitions
#   MAX_R_SAMPLES = 100: Sample 100 candidate radius values
# =============================================================================

K = 5
CAPACITY = 250
MAX_PARTITIONS = 10
MAX_R_SAMPLES = 100

print(f"{'='*70}")
print(f"{'EXPERIMENT CONFIGURATION':^70}")
print(f"{'='*70}")
print(f"  |C| = {len(C)} clients (bank customers)")
print(f"  |F| = {len(F)} candidate facilities")
print(f"  k = {K} facilities to open")
print(f"  U = {CAPACITY} capacity per facility")
print(f"  Total capacity = {K * CAPACITY}")
print(f"  Features: age, balance, duration (3D Euclidean, normalized)")
print(f"{'='*70}\n")

if K * CAPACITY < len(C):
    print(f"WARNING: Total capacity ({K * CAPACITY}) < clients ({len(C)})")
    print(f"  The problem is INFEASIBLE. Increase K or CAPACITY.")
else:
    print(f"Feasibility check PASSED: {K * CAPACITY} ≥ {len(C)}")
    print(f"  The problem may have a solution.\n")

solution = capacitated_k_center(
    C=C,
    F=F,
    k=K,
    capacity=CAPACITY,
    max_partitions=MAX_PARTITIONS,
    max_r_samples=MAX_R_SAMPLES
)


In [ ]:
# =============================================================================
# SECTION 10: RESULTS ANALYSIS
# =============================================================================
# Analyze the solution returned by the algorithm.
# Distance units are Euclidean (normalized) — not miles.
# =============================================================================

if solution:
    distances = solution['all_distances']

    print("\n" + "="*70)
    print(f"{'SOLUTION ANALYSIS':^70}")
    print("="*70)

    print(f"\n📏 DISTANCE METRICS (Euclidean, normalized units):")
    print(f"   Maximum:  {np.max(distances):.4f}  ← SERVICE RADIUS (minimized)")
    print(f"   Average:  {np.mean(distances):.4f}")
    print(f"   Median:   {np.median(distances):.4f}")
    print(f"   Minimum:  {np.min(distances):.4f}")
    print(f"   Std Dev:  {np.std(distances):.4f}")

    print(f"\n📋 SOLUTION DETAILS:")
    print(f"   Partition used: {solution['partition']}")
    print(f"     → Interpretation: k_i facilities opened near each initial center")
    print(f"   Total slots: {solution['total_slots']} (should equal k={K})")
    print(f"   Facilities opened: {solution['num_facilities']}")
    print(f"   Facility indices in F: {solution['facility_indices']}")

    print(f"\n🏥 PER-FACILITY BREAKDOWN:")
    print(f"   {'Facility':<12} {'Clients':<10} {'% Capacity':<12} {'Status':<8} {'Location (normalized)'}")
    print(f"   {'-'*70}")
    for f_idx in sorted(solution['facility_indices']):
        load = solution['facility_loads'].get(f_idx, 0)
        pct = load / CAPACITY * 100
        status = "✓ OK" if load <= CAPACITY else "✗ OVER"
        coords = F[f_idx]
        coord_str = f"({coords[0]:.2f}, {coords[1]:.2f}, {coords[2]:.2f})"
        print(f"   F[{f_idx}]      {load:<10} {pct:>6.1f}%       {status}     {coord_str}")

    print(f"\n✅ CONSTRAINT VERIFICATION:")
    print(f"   [Constraint 1] At most k={K} facilities opened: {solution['num_facilities']} ≤ {K} → ✓ SATISFIED")
    print(f"   [Constraint 2] Capacity U={CAPACITY} not exceeded: max={max(solution['facility_loads'].values())} ≤ {CAPACITY} → ✓ SATISFIED")
    print(f"   [Constraint 3] All clients assigned: {len(solution['assignments'])} = {len(C)} → ✓ SATISFIED")

    print(f"\n{'='*70}")
else:
    print("No feasible solution found. Try increasing k or capacity.")


In [ ]:
# =============================================================================
# SECTION 11: VISUALIZATION (2D projections of 3D solution)
# =============================================================================
# Since the data is 3D (age, balance, duration), we create two 2D views:
#   Left plot:  Age (x) vs Balance (y)
#   Right plot: Age (x) vs Duration (y)
#
# Each color represents clients assigned to one facility.
# Stars mark the opened facility locations.
# =============================================================================

if solution:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    num_fac = len(solution['facility_indices'])
    colors = plt.cm.tab10(np.linspace(0, 1, max(num_fac, 1)))
    fac_to_color = {f: i for i, f in enumerate(solution['facility_indices'])}

    # ----- Plot 1: Age vs Balance -----
    ax = axes[0]
    for c_idx, f_idx in solution['assignments'].items():
        color_idx = fac_to_color.get(f_idx, 0)
        ax.scatter(C[c_idx, 0], C[c_idx, 1],
                   c=[colors[color_idx]], s=20, alpha=0.5)

    for i, f_idx in enumerate(solution['facility_indices']):
        load = solution['facility_loads'].get(f_idx, 0)
        ax.scatter(F[f_idx, 0], F[f_idx, 1],
                   c=[colors[i]], s=300, marker='*', edgecolors='black',
                   linewidths=2, label=f'F[{f_idx}] ({load} clients)', zorder=5)

    ax.set_xlabel('Age (normalized)', fontsize=11)
    ax.set_ylabel('Balance (normalized)', fontsize=11)
    ax.set_title('Age vs Balance', fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

    # ----- Plot 2: Age vs Duration -----
    ax = axes[1]
    for c_idx, f_idx in solution['assignments'].items():
        color_idx = fac_to_color.get(f_idx, 0)
        ax.scatter(C[c_idx, 0], C[c_idx, 2],
                   c=[colors[color_idx]], s=20, alpha=0.5)

    for i, f_idx in enumerate(solution['facility_indices']):
        load = solution['facility_loads'].get(f_idx, 0)
        ax.scatter(F[f_idx, 0], F[f_idx, 2],
                   c=[colors[i]], s=300, marker='*', edgecolors='black',
                   linewidths=2, label=f'F[{f_idx}] ({load} clients)', zorder=5)

    ax.set_xlabel('Age (normalized)', fontsize=11)
    ax.set_ylabel('Duration (normalized)', fontsize=11)
    ax.set_title('Age vs Duration', fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

    fig.suptitle(
        f'Bank Marketing: Capacitated K-Center Solution\n'
        f'k={K}, U={CAPACITY}, n={len(C)} | '
        f'Max Distance: {solution["actual_max_distance"]:.3f} (Euclidean, normalized)',
        fontsize=13, fontweight='bold'
    )

    plt.tight_layout()
    plt.savefig('bank_marketing_clustering.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved: bank_marketing_clustering.png")

    if solution.get('slot_to_facility'):
        print("\nSlot → Facility Mapping (from Line 16):")
        for slot, fac in sorted(solution['slot_to_facility'].items()):
            coords = F[fac]
            print(f"   {slot} → F[{fac}] at ({coords[0]:.4f}, {coords[1]:.4f}, {coords[2]:.4f})")
else:
    print("No solution to visualize.")


In [ ]:
# =============================================================================
# GRAPH 1: Cost vs k (Main Graph)
# =============================================================================
# X-axis: k (number of facilities)
# Y-axis: Max distance (k-center cost)
# Lines: Capacitated (U=250) vs Uncapacitated (U=∞)
# =============================================================================

k_values = [3, 4, 5, 6, 8, 10, 12]

capacitated_costs = []
uncapacitated_costs = []

print("Running experiments...")
for k in k_values:
    print(f"  k = {k}...", end=" ")

    # Capacitated (U = 250)
    if k * CAPACITY >= len(C):
        sol_cap = capacitated_k_center(
            C, F, k,
            capacity=CAPACITY,
            max_partitions=15,
            max_r_samples=100,
            verbose=False
        )
        cap_cost = sol_cap['actual_max_distance'] if sol_cap else np.nan
    else:
        cap_cost = np.nan

    # Uncapacitated (U = n, effectively no limit)
    sol_uncap = capacitated_k_center(
        C, F, k,
        capacity=len(C),
        max_partitions=15,
        max_r_samples=100,
        verbose=False
    )
    uncap_cost = sol_uncap['actual_max_distance'] if sol_uncap else np.nan

    capacitated_costs.append(cap_cost)
    uncapacitated_costs.append(uncap_cost)

    if np.isnan(cap_cost):
        print(f"Capacitated Cost=nan")
    else:
        print(f"Capacitated Cost={cap_cost:.2f}")

# -----------------------------------------------------------------------------
# Plot
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5), dpi=120)

ax.plot(k_values, capacitated_costs,
        color='#e74c3c', marker='o', markersize=8, linewidth=2,
        label=f'Capacitated (U={CAPACITY})')

ax.plot(k_values, uncapacitated_costs,
        color='#3498db', marker='s', markersize=8, linewidth=2,
        label='Uncapacitated')

ax.set_xlabel('Number of Facilities (k)', fontsize=12, fontweight='bold')
ax.set_ylabel('Max Distance (Euclidean, normalized)', fontsize=12, fontweight='bold')
ax.set_title('Capacitated k-Center: Cost vs k', fontsize=14, fontweight='bold')

ax.set_xticks(k_values)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('bank_cost_vs_k.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Saved: bank_cost_vs_k.png")


In [ ]:
# =============================================================================
# GRAPH 1: Cost vs k (Multiple Capacity Values)
# =============================================================================
# X-axis: k (number of facilities)
# Y-axis: Max distance (k-center cost, Euclidean normalized)
# Lines: Different capacity values U
#
# KEY INSIGHT: As capacity increases, the cost curve drops toward
# the uncapacitated baseline. Very tight capacity makes small k
# values INFEASIBLE (k × U < n).
# =============================================================================

k_values = [3, 4, 5, 6, 8, 10]
capacity_values = [150, 250, 500, len(C)]

print(f"{'='*70}")
print(f"{'COST vs k (VARYING CAPACITY)':^70}")
print(f"{'='*70}")
print(f"  k values: {k_values}")
print(f"  n = {len(C)} clients")
print()

results = {}
cap_labels = []

for U_val in capacity_values:
    label = f'U={U_val}' if U_val < len(C) else 'Uncapacitated'
    cap_labels.append(label)
    results[label] = []

    for k in k_values:
        if k * U_val < len(C):
            results[label].append(np.nan)
            print(f"  k={k:>2}, {label:>15}: INFEASIBLE (k×U={k*U_val} < {len(C)})")
            continue

        sol = capacitated_k_center(
            C, F, k,
            capacity=U_val,
            max_partitions=10,
            max_r_samples=80,
            verbose=False
        )
        cost = sol['actual_max_distance'] if sol else np.nan
        results[label].append(cost)
        print(f"  k={k:>2}, {label:>15}: Cost = {cost:.4f}")

# -------------------------------------------------------------------------
# Summary Table
# -------------------------------------------------------------------------
print(f"\n{'='*70}")
print(f"{'SUMMARY TABLE: Cost vs k for Bank Marketing':^70}")
print(f"{'='*70}")
header_str = f"  {'k':>4}"
for label in cap_labels:
    header_str += f"  {label:>14}"
print(header_str)
print(f"  {'-'*66}")
for ki, k in enumerate(k_values):
    row_str = f"  {k:>4}"
    for label in cap_labels:
        val = results[label][ki]
        if np.isnan(val):
            row_str += f"  {'INFEASIBLE':>14}"
        else:
            row_str += f"  {val:>14.4f}"
    print(row_str)
print(f"{'='*70}")

# -------------------------------------------------------------------------
# Plot
# -------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 6), dpi=120)

line_styles = ['-o', '-s', '-^', '-D']
line_colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']

for i, label in enumerate(cap_labels):
    costs = results[label]
    valid_k = [k_values[j] for j in range(len(k_values)) if not np.isnan(costs[j])]
    valid_c = [costs[j] for j in range(len(costs)) if not np.isnan(costs[j])]
    if valid_k:
        ax.plot(valid_k, valid_c,
                line_styles[i % len(line_styles)],
                color=line_colors[i % len(line_colors)],
                markersize=8, linewidth=2, label=label)

ax.set_xlabel('Number of Facilities (k)', fontsize=12, fontweight='bold')
ax.set_ylabel('Max Distance (Euclidean, normalized)', fontsize=12, fontweight='bold')
ax.set_title('Bank Marketing: Cost vs k for Different Capacities',
             fontsize=14, fontweight='bold')
ax.set_xticks(k_values)
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('bank_cost_vs_k_capacity.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nSaved: bank_cost_vs_k_capacity.png")


In [ ]:
# =============================================================================
# SOLUTION VERIFICATION
# =============================================================================

if solution:
    print("SOLUTION VERIFICATION")
    print("-"*70)
    print(f"  ✓ Dataset:               Bank Marketing (Moro et al., 2014)")
    print(f"  ✓ Features:              age, balance, duration (3D Euclidean)")
    print(f"  ✓ Distance metric:       Euclidean (Z-score normalized)")
    print(f"  ✓ Sample size:           {len(C)} points")
    print(f"  ✓ Partition used:        {solution['partition']}")
    print(f"  ✓ Total slots created:   {solution['total_slots']} = k")
    print(f"  ✓ Facilities opened:     {solution['num_facilities']} ≤ k")
    print(f"  ✓ Clients assigned:      {len(solution['assignments'])} = |C|")
    print(f"  ✓ Max facility load:     {max(solution['facility_loads'].values())} ≤ U")
    print(f"  ✓ Maximum distance:      {solution['actual_max_distance']:.4f} (Euclidean)")
